# 18B — SHARP Cycle-24 Freeze, Calibration and Threshold Selection

Purpose: freeze the SHARP-only reference pipeline using Cycle-24 only.

Roles:
- `cycle24_final_refit_pool`: model training
- `cycle24_calibration_holdout`: probability calibration
- `cycle24_threshold_holdout`: threshold selection

Cycle-25 remains untouched.

Inputs: three SHARP steps at t−288, t−192 and t−96 min, 15 features each, flattened to 45 ordered inputs.

Quality rule: all three records require QUALITY=0, 15/15 finite features, one exact source match, no explicit NOAA conflict, and three unique exact records.

Models carried from 18A: Logistic Regression and Random Forest.

Calibration: Platt/sigmoid calibration fitted only on the calibration holdout.

Threshold: TSS-maximising threshold selected only on the threshold holdout.

This is a Cycle-24 pipeline-freezing step, not a Cycle-25 performance result.


In [ ]:
from pathlib import Path
import ast, gzip, json, time
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
    roc_curve, log_loss
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

BASE = Path.home() / "aia_sharp_cycle24_input" / "20260916T195026327454Z"
OUT = Path.home() / "sharp_cycle24_freeze_20260917"
(OUT / "models").mkdir(parents=True, exist_ok=True)
(OUT / "predictions").mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 20260917
EPS = 1e-6
print("BASE:", BASE)
print("OUT:", OUT)


## 1. Load arrays and roles

In [ ]:
X3 = np.load(BASE / "sharp_raw_three_slot.npy", mmap_mode="r")
finite = np.load(BASE / "sharp_finite_mask.npy", mmap_mode="r")
matches = np.load(BASE / "sharp_record_match_counts.npy", mmap_mode="r")
y = np.load(BASE / "original_manifest_targets.npy", mmap_mode="r")
rows = pd.read_csv(BASE / "cycle24_array_rows.csv.gz")

assert X3.shape == (64725, 3, 15)
assert finite.shape == X3.shape
assert matches.shape == (64725, 3)
assert y.shape == (64725,)
assert len(rows) == 64725

X45 = np.asarray(X3).reshape(len(X3), 45)
print(rows["proposed_final_role"].value_counts().to_string())


## 2. Apply the same conservative quality rule used in 18A

In [ ]:
source_quality = {}
with gzip.open(BASE / "unique_sharp_source_records.jsonl.gz", "rt") as handle:
    for line in handle:
        rec = json.loads(line)
        idx = int(rec["slot_source_index"])
        q = rec["raw_QUALITY"]
        q0 = (len(q) == 1 and str(q[0]).strip() == "0") if isinstance(q, list) else str(q).strip() == "0"
        source_quality[idx] = {
            "quality_zero": q0,
            "finite_feature_count": int(rec["finite_feature_count"]),
            "match_count": int(rec["match_count"]),
            "noaa_conflict": bool(rec["noaa_conflict"]),
        }

quality_pass = np.zeros(len(rows), dtype=bool)
for i, raw_indices in enumerate(rows["slot_source_indices"]):
    inds = ast.literal_eval(raw_indices)
    if len(inds) != 3:
        continue
    quality_pass[i] = all(
        (info := source_quality.get(int(idx))) is not None
        and info["quality_zero"]
        and info["finite_feature_count"] == 15
        and info["match_count"] == 1
        and not info["noaa_conflict"]
        for idx in inds
    )

integrity_pass = (
    finite.all(axis=(1, 2))
    & (matches == 1).all(axis=1)
    & (rows["three_unique_exact_records"].to_numpy() == 1)
    & (rows["raw_numeric_complete"].to_numpy() == 1)
)

rows = rows.copy()
rows["eligible"] = quality_pass & integrity_pass
rows["array_label"] = y
print(rows[rows["eligible"]].groupby("proposed_final_role").agg(
    n=("array_row","size"),
    positives=("array_label","sum"),
    regions=("region_component_id","nunique")
).to_string())


## 3. Verify role separation

In [ ]:
TRAIN_ROLE = "cycle24_final_refit_pool"
CAL_ROLE = "cycle24_calibration_holdout"
THR_ROLE = "cycle24_threshold_holdout"

train_df = rows[(rows["eligible"]) & (rows["proposed_final_role"] == TRAIN_ROLE)].copy()
cal_df = rows[(rows["eligible"]) & (rows["proposed_final_role"] == CAL_ROLE)].copy()
thr_df = rows[(rows["eligible"]) & (rows["proposed_final_role"] == THR_ROLE)].copy()

role_sets = {
    "train": set(train_df["region_component_id"]),
    "calibration": set(cal_df["region_component_id"]),
    "threshold": set(thr_df["region_component_id"]),
}
for a, sa in role_sets.items():
    for b, sb in role_sets.items():
        if a < b:
            overlap = sa & sb
            print(a, "vs", b, "overlap_regions =", len(overlap))
            assert not overlap

def extract(df):
    idx = df["array_row"].to_numpy(dtype=int)
    return X45[idx], y[idx], idx

X_train, y_train, idx_train = extract(train_df)
X_cal, y_cal, idx_cal = extract(cal_df)
X_thr, y_thr, idx_thr = extract(thr_df)

for name, Xp, yp in [("train",X_train,y_train),("calibration",X_cal,y_cal),("threshold",X_thr,y_thr)]:
    assert np.isfinite(Xp).all()
    print(name, Xp.shape, "positives:", int(yp.sum()))
print("STATUS: ROLE_SEPARATION_VERIFIED_NO_CYCLE25_USED")


## 4. Helpers

In [ ]:
def tss_from_cm(tn, fp, fn, tp):
    tpr = tp/(tp+fn) if tp+fn else np.nan
    fpr = fp/(fp+tn) if fp+tn else np.nan
    return tpr-fpr

def hss_from_cm(tn, fp, fn, tp):
    num = 2*(tp*tn-fn*fp)
    den = (tp+fn)*(fn+tn)+(tp+fp)*(fp+tn)
    return num/den if den else np.nan

def threshold_metrics(y_true, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    return {
        "threshold": float(thr), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "tss": float(tss_from_cm(tn,fp,fn,tp)), "hss": float(hss_from_cm(tn,fp,fn,tp)),
        "precision": float(precision_score(y_true,pred,zero_division=0)),
        "recall": float(recall_score(y_true,pred,zero_division=0)),
        "f1": float(f1_score(y_true,pred,zero_division=0)),
    }

def best_tss_threshold(y_true, p):
    fpr, tpr, thr = roc_curve(y_true, p)
    score = tpr-fpr
    valid = np.where(np.isfinite(thr))[0]
    i = valid[np.argmax(score[valid])]
    return float(thr[i])

def logit(p):
    p = np.clip(np.asarray(p), EPS, 1-EPS)
    return np.log(p/(1-p)).reshape(-1,1)

def fit_platt(y_true, raw_p):
    c = LogisticRegression(solver="lbfgs", max_iter=5000, random_state=RANDOM_STATE)
    c.fit(logit(raw_p), y_true)
    return c

def apply_platt(c, raw_p):
    return c.predict_proba(logit(raw_p))[:,1]

def prob_metrics(y_true, p):
    return {
        "roc_auc": float(roc_auc_score(y_true,p)),
        "pr_auc": float(average_precision_score(y_true,p)),
        "brier": float(brier_score_loss(y_true,p)),
        "log_loss": float(log_loss(y_true,np.clip(p,EPS,1-EPS))),
    }


## 5. Train base models on training pool only

In [ ]:
models = {
    "logistic_regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=5000, solver="lbfgs", random_state=RANDOM_STATE)),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=500, class_weight="balanced_subsample",
        max_features="sqrt", min_samples_leaf=2, n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

base_models, raw = {}, {}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    base_models[name] = model
    raw[name] = {
        "cal": model.predict_proba(X_cal)[:,1],
        "thr": model.predict_proba(X_thr)[:,1],
        "fit_seconds": time.time()-t0,
    }
    print(name, "fit_seconds:", raw[name]["fit_seconds"])


## 6. Calibrate on calibration holdout, select threshold on threshold holdout

In [ ]:
records = []
for name, model in base_models.items():
    calibrator = fit_platt(y_cal, raw[name]["cal"])
    p_cal = apply_platt(calibrator, raw[name]["cal"])
    p_thr = apply_platt(calibrator, raw[name]["thr"])
    selected_thr = best_tss_threshold(y_thr, p_thr)

    rec = {
        "model": name,
        "fit_seconds": float(raw[name]["fit_seconds"]),
        "calibration_raw": prob_metrics(y_cal, raw[name]["cal"]),
        "calibration_calibrated": prob_metrics(y_cal, p_cal),
        "threshold_holdout_probability_metrics": prob_metrics(y_thr, p_thr),
        "selected_threshold": selected_thr,
        "threshold_holdout_classification": threshold_metrics(y_thr, p_thr, selected_thr),
    }
    records.append(rec)

    joblib.dump(model, OUT / "models" / f"{name}__base_model.joblib")
    joblib.dump(calibrator, OUT / "models" / f"{name}__platt_calibrator.joblib")

    for role_name, df, yy, raw_p, cal_p in [
        ("calibration", cal_df, y_cal, raw[name]["cal"], p_cal),
        ("threshold", thr_df, y_thr, raw[name]["thr"], p_thr),
    ]:
        pred = df[["target_sample_id","array_row","region_component_id","stored_year"]].copy()
        pred["y_true"] = yy
        pred["raw_probability"] = raw_p
        pred["calibrated_probability"] = cal_p
        pred["selected_threshold"] = selected_thr
        pred["prediction"] = (cal_p >= selected_thr).astype(int)
        pred.to_csv(OUT / "predictions" / f"{name}__{role_name}.csv.gz", index=False, compression="gzip")

summary = pd.json_normalize(records)
summary.to_csv(OUT / "cycle24_freeze_summary.csv", index=False)
with open(OUT / "cycle24_freeze_summary.json","w") as f:
    json.dump(records,f,indent=2)

protocol = {
    "status": "CYCLE24_REFERENCE_PIPELINE_FROZEN_PENDING_INDEPENDENT_CYCLE25_EVALUATION",
    "cycle25_used": False,
    "supplementary_2026_used": False,
    "train_role": TRAIN_ROLE,
    "calibration_role": CAL_ROLE,
    "threshold_role": THR_ROLE,
    "models": list(base_models.keys()),
    "calibration_method": "Platt/sigmoid fitted only on calibration holdout",
    "threshold_method": "TSS-maximising threshold selected only on threshold holdout after calibration",
    "scientific_clearance": False,
}
with open(OUT / "protocol_record.json","w") as f:
    json.dump(protocol,f,indent=2)

print(summary.to_string(index=False))
print("OUTPUT:", OUT)
print("STATUS: CYCLE24_SHARP_REFERENCE_FROZEN_NO_CYCLE25_USED")
